In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.data_loading import data_loading
from biked_commons.validation.bike_bench_validation_functions import bike_bench_validation_functions
from biked_commons.validation.base_validation_function import construct_tensor_validator, construct_dataframe_validator

This builds the overall validation function

Read in the Bike Data

In [2]:
data_train = data_loading.load_bike_bench_train()
data_test = data_loading.load_bike_bench_test()
data = pd.concat([data_train, data_test], ignore_index=True)
display(data)

,CS textfield,BB textfield,Stack,Head angle,Head tube length textfield,Seat stay junction0,Seat tube length,Seat angle,DT Length,FORK0R,...,Handlebar style OHCLASS: 2,Stem kind OHCLASS: 0,Stem kind OHCLASS: 1,Stem kind OHCLASS: 2,Fork type OHCLASS: 0,Fork type OHCLASS: 1,Fork type OHCLASS: 2,Seat tube type OHCLASS: 0,Seat tube type OHCLASS: 1,Seat tube type OHCLASS: 2
0,420.00,25.0,536.36066,70.0,112.8,45.0,400.0,73.0,648.36316,45.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,796.94,-3.0,363.41064,39.0,181.2,36.0,567.0,37.0,1243.38900,45.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,458.00,50.0,549.67426,63.5,159.9,45.0,460.0,76.0,742.93770,45.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,403.00,70.0,569.91583,73.0,134.0,258.0,516.8,79.0,627.59420,45.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,440.00,70.0,574.76190,74.0,125.2,45.0,579.6,74.0,651.83970,45.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4495,430.00,65.0,630.07196,72.5,194.1,60.0,550.3,73.5,639.51460,65.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4496,385.00,47.0,527.72925,74.0,128.7,50.0,520.0,76.0,617.48610,30.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4497,515.00,35.0,577.54860,65.0,117.0,11.0,550.2,66.0,662.99440,85.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
4498,435.00,20.0,527.21875,72.0,130.0,65.0,415.0,72.0,608.89920,45.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


This builds the overall validation function

In [3]:
dataframe_validator = construct_dataframe_validator(bike_bench_validation_functions)

Call the validation function and check the results. Remember, a score less or equal to 0 is valid. The more postive, the greater the magnitude of constraint violation. The more negative the further away from violating the constraint. A design is overall valid if all constraints are less or equal to 0

In [4]:
validity_scores = dataframe_validator(data)
display(validity_scores.head(10)) #design 6's saddle height is too small, for example

,Saddle height too small,Seat post too short,Head tube lower extension longer than head tube,Head tube upper extension and lower extension overlap,Strictly postive parameter is negative,Chain stay smaller than rear wheel radius,Chain stay shorter than BB drop,Seat stay smaller than rear wheel radius,Seat Tube Intersects Rear Wheel2,Down tube can't reach head tube,Rear wheel cutout severs seat tube,Foot intersects front wheel,Crank hits ground in lowest position,RGB value greater than 255,Chain stays intersect,Tube wall thickness exceeds radius,Down tube intersects front wheel
0,-278.000000,-42.000000,-69.600006,-43.200005,0.0,-85.000000,-395.000000,-110.443512,-42.726562,-177.405121,-9.999997e+08,-5.256023e+04,-122.500000,-612.0,-7.325001,0.0,-106.741547
1,-36.200012,-283.799988,-134.199997,-66.199997,0.0,-454.040009,-799.940002,-150.122528,-121.703033,-964.433228,-9.999997e+08,-1.794022e+06,-158.399994,-663.0,0.924999,0.0,-152.451324
2,-240.000000,-230.000000,-69.899994,-41.599991,0.0,-85.500000,-408.000000,-127.807098,-39.643219,-255.819336,-9.999997e+08,-1.916954e+05,-135.000000,-306.0,-9.000000,0.0,-113.410950
3,-199.200012,-120.799988,-100.900002,-72.699997,0.0,-66.000000,-333.000000,-56.222290,-6.500000,-154.169495,-4.872574e+01,-2.010055e+03,-79.500000,-255.0,-7.325001,0.0,-38.925415
4,-98.400024,-191.599976,-81.500000,-50.500000,0.0,-103.000000,-370.000000,-191.612305,-47.273804,-155.420654,-9.999997e+08,-7.866961e+03,-79.500000,-714.0,-10.250000,0.0,-54.127808
5,-87.599976,-232.400024,-109.300003,-75.099998,0.0,-70.500000,-355.000000,-198.786499,-14.804443,-165.845276,-9.999997e+08,-3.344159e+04,-97.000000,-510.0,-7.325001,0.0,-82.431091
6,-95.799988,-224.200012,-99.600006,-67.700005,0.0,-68.000000,-335.000000,-165.000214,-6.492798,-141.723480,-9.999997e+08,5.116047e+03,-79.500000,-612.0,-7.325001,0.0,-45.238892
7,564.500000,-854.500000,-86.900002,-57.900002,0.0,-52.339996,-339.339996,-206.046387,-9.998383,-169.684875,-9.999997e+08,2.263673e+04,-99.500000,0.0,-9.000000,0.0,-8.609314
8,-50.000000,-240.000000,-75.000000,-52.000000,0.0,-60.000000,-365.000000,-56.705353,-14.608917,-164.358459,-9.999997e+08,-2.850275e+04,-117.500000,-765.0,-5.325001,0.0,-86.747528
9,-15.000000,-275.000000,-60.900002,-40.000000,0.0,-56.799988,-356.799988,-63.294312,-23.436981,-156.791382,-9.999997e+08,-2.596281e+04,-112.500000,-306.0,-6.500000,0.0,-109.988831


We can check how many designs are valid overall:

In [5]:
#overall valid if all constraints are less or equal to 0

validity = np.all(validity_scores <= 0, axis=1)
print(validity[:10])
print(f"Fraction of valid designs: {np.sum(validity) / len(validity)}")

0     True
1    False
2     True
3     True
4     True
5     True
6    False
7    False
8     True
9     True
dtype: bool
Fraction of valid designs: 0.5755555555555556


If we want to calculate gradients, we need to use the pytorch version of the validator. 

In [6]:
COLUMN_NAMES = list(data.columns)
tensor_validator, validation_names = construct_tensor_validator(bike_bench_validation_functions, COLUMN_NAMES)

In [7]:
print(validation_names)

['Saddle height too small', 'Seat post too short', 'Head tube lower extension longer than head tube', 'Head tube upper extension and lower extension overlap', 'Strictly postive parameter is negative', 'Chain stay smaller than rear wheel radius', 'Chain stay shorter than BB drop', 'Seat stay smaller than rear wheel radius', 'Seat Tube Intersects Rear Wheel2', "Down tube can't reach head tube", 'Rear wheel cutout severs seat tube', 'Foot intersects front wheel', 'Crank hits ground in lowest position', 'RGB value greater than 255', 'Chain stays intersect', 'Tube wall thickness exceeds radius', 'Down tube intersects front wheel']


In [8]:
data_tensor = torch.tensor(data.values, dtype=torch.float32, requires_grad=True)
validity_scores_tens = tensor_validator(data_tensor)
print(validity_scores_tens) # should be the same as the dataframe version

loss = validity_scores_tens[validity_scores_tens > 0].sum() # we make our objective function equal to the sum of constraint violation
loss.backward()
gradients = data_tensor.grad
gradient_df = pd.DataFrame(gradients.numpy(), columns=COLUMN_NAMES, index = data.index)

tensor([[-278.0000,  -42.0000,  -69.6000,  ...,   -7.3250,    0.0000,
         -106.7415],
        [ -36.2000, -283.8000, -134.2000,  ...,    0.9250,    0.0000,
         -152.4513],
        [-240.0000, -230.0000,  -69.9000,  ...,   -9.0000,    0.0000,
         -113.4109],
        ...,
        [ -94.8000, -145.2000,  -85.3000,  ...,  -19.1750,    0.0000,
         -180.3811],
        [ -95.0000, -195.0000,  -70.0000,  ...,   -5.0000,    0.0000,
          -74.7780],
        [-118.0000, -172.0000, -188.1000,  ...,  -10.5000,    0.0000,
          -54.5861]], grad_fn=<CopySlices>)


The gradients tell us how to adjust the parameters to ideally fix the problem

In [9]:
display(gradient_df.head(10)) # we can see that there are some gradients in the seat tube length column, for example

,CS textfield,BB textfield,Stack,Head angle,Head tube length textfield,Seat stay junction0,Seat tube length,Seat angle,DT Length,FORK0R,...,Handlebar style OHCLASS: 2,Stem kind OHCLASS: 0,Stem kind OHCLASS: 1,Stem kind OHCLASS: 2,Fork type OHCLASS: 0,Fork type OHCLASS: 1,Fork type OHCLASS: 2,Seat tube type OHCLASS: 0,Seat tube type OHCLASS: 1,Seat tube type OHCLASS: 2
0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,105.688675,571.713135,4765.282227,-548.169861,0.0,0.0,0.0,-967.582886,-634.949341,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,91.740173,448.767944,4074.546143,-434.473633,0.0,1.0,0.0,-808.168640,-569.083862,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Test gradient descent to see if validity improves

In [10]:
updated_data = data
step_size = 0.1
for steps in trange(1000):
    #calculate gradients
    data_tensor = torch.tensor(updated_data.values, dtype=torch.float32, requires_grad=True)
    validity_scores_tens = tensor_validator(data_tensor)
    loss = validity_scores_tens[validity_scores_tens > 0].sum()
    loss.backward()
    gradients = data_tensor.grad
    gradients = gradients.numpy()
    updated_data = updated_data - step_size * gradients

100%|██████████| 1000/1000 [00:22<00:00, 44.21it/s]


In [11]:
updated_data.head(10)

,CS textfield,BB textfield,Stack,Head angle,Head tube length textfield,Seat stay junction0,Seat tube length,Seat angle,DT Length,FORK0R,...,Handlebar style OHCLASS: 2,Stem kind OHCLASS: 0,Stem kind OHCLASS: 1,Stem kind OHCLASS: 2,Fork type OHCLASS: 0,Fork type OHCLASS: 1,Fork type OHCLASS: 2,Seat tube type OHCLASS: 0,Seat tube type OHCLASS: 1,Seat tube type OHCLASS: 2
0,420.00,25.000000,536.360660,70.000000,112.800000,45.0,400.000000,73.0,648.36316,45.000000,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,796.94,-3.000000,363.410640,39.000000,181.200000,36.0,567.000000,37.0,1243.38900,45.000000,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,458.00,50.000000,549.674260,63.500000,159.900000,45.0,460.000000,76.0,742.93770,45.000000,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,403.00,70.000000,569.915830,73.000000,134.000000,258.0,516.800000,79.0,627.59420,45.000000,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,440.00,70.000000,574.761900,74.000000,125.200000,45.0,579.600000,74.0,651.83970,45.000000,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,415.00,60.000000,627.335300,73.000000,170.000000,55.0,625.400000,73.0,688.65650,45.000000,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
6,405.00,36.269011,662.687269,3.765314,370.960231,45.0,582.200000,72.5,639.46709,228.012444,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
7,389.34,69.327218,444.486447,-636.979540,219.473194,45.0,495.499999,75.5,780.83724,125.870461,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
8,395.00,30.000000,532.084170,70.500000,100.000000,15.0,300.000000,72.5,625.74450,45.000000,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
9,356.80,0.000000,477.103700,71.000000,100.400000,45.0,285.000000,72.0,576.31300,45.000000,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


We should see that the fraciton of valid designs increases.

In [13]:
#check validity of updated data
validity_scores = dataframe_validator(updated_data)
validity = np.all(validity_scores <= 0, axis=1)
print(validity.mean())

0.8135555555555556


In [14]:
display(validity_scores.head(10))


,Saddle height too small,Seat post too short,Head tube lower extension longer than head tube,Head tube upper extension and lower extension overlap,Strictly postive parameter is negative,Chain stay smaller than rear wheel radius,Chain stay shorter than BB drop,Seat stay smaller than rear wheel radius,Seat Tube Intersects Rear Wheel2,Down tube can't reach head tube,Rear wheel cutout severs seat tube,Foot intersects front wheel,Crank hits ground in lowest position,RGB value greater than 255,Chain stays intersect,Tube wall thickness exceeds radius,Down tube intersects front wheel
0,-278.000000,-42.000000,-69.600006,-43.200005,0.000000,-85.000000,-395.000000,-110.443512,-42.726562,-177.405121,-9.999997e+08,-5.256023e+04,-122.500000,-612.000000,-7.325001,0.0,-1.067415e+02
1,-36.200012,-283.799988,-134.199997,-66.199997,0.000000,-454.040009,-799.940002,-150.122528,-121.703033,-964.433228,-9.999997e+08,-1.794022e+06,-158.399994,-663.000000,-0.125000,0.0,-1.524513e+02
2,-240.000000,-230.000000,-69.899994,-41.599991,0.000000,-85.500000,-408.000000,-127.807098,-39.643219,-255.819336,-9.999997e+08,-1.916954e+05,-135.000000,-306.000000,-9.000000,0.0,-1.134109e+02
3,-199.200012,-120.799988,-100.900002,-72.699997,0.000000,-66.000000,-333.000000,-56.222290,-6.500000,-154.169495,-4.872574e+01,-2.010055e+03,-79.500000,-255.000000,-7.325001,0.0,-3.892542e+01
4,-98.400024,-191.599976,-81.500000,-50.500000,0.000000,-103.000000,-370.000000,-191.612305,-47.273804,-155.420654,-9.999997e+08,-7.866961e+03,-79.500000,-714.000000,-10.250000,0.0,-5.412781e+01
5,-87.599976,-232.400024,-109.300003,-75.099998,0.000000,-70.500000,-355.000000,-198.786499,-14.804443,-165.845276,-9.999997e+08,-3.344159e+04,-97.000000,-510.000000,-7.325001,0.0,-8.243109e+01
6,-95.799988,-224.200012,-441.420471,-409.520447,70.460228,-67.991638,-368.730988,-196.961609,-20.888763,-5.767883,-9.999997e+08,-1.528784e+08,-113.239349,-611.900024,-7.325001,0.0,-1.294130e+04
7,364.500000,-654.500000,-256.846375,-227.846390,674.352722,-52.339996,-320.012787,-118.667847,-2.256622,-591.293823,-9.999997e+08,-3.228132e+05,-80.172791,0.000000,-9.000000,0.0,-3.039115e+08
8,-50.000000,-240.000000,-75.000000,-52.000000,0.000000,-60.000000,-365.000000,-56.705353,-14.608917,-164.358459,-9.999997e+08,-2.850275e+04,-117.500000,-765.000000,-5.325001,0.0,-8.674753e+01
9,-15.000000,-275.000000,-60.900002,-40.000000,0.000000,-56.799988,-356.799988,-63.294312,-23.436981,-156.791382,-9.999997e+08,-2.596281e+04,-112.500000,-306.000000,-6.500000,0.0,-1.099888e+02


In [15]:
margin = 0.05
parameter_ranges = np.max(data.values, axis=0) - np.min(data.values, axis=0)
lower_bounds = np.min(data.values, axis=0)-margin*parameter_ranges
upper_bounds = np.max(data.values, axis=0)+margin*parameter_ranges

#randomly sample a design
random_design = np.random.uniform(lower_bounds, upper_bounds)

#check validity of random design

validity_scores = dataframe_validator(pd.DataFrame(random_design.reshape(1, -1), columns=COLUMN_NAMES))
display(validity_scores) # should be all less than or equal to 0


,Saddle height too small,Seat post too short,Head tube lower extension longer than head tube,Head tube upper extension and lower extension overlap,Strictly postive parameter is negative,Chain stay smaller than rear wheel radius,Chain stay shorter than BB drop,Seat stay smaller than rear wheel radius,Seat Tube Intersects Rear Wheel2,Down tube can't reach head tube,Rear wheel cutout severs seat tube,Foot intersects front wheel,Crank hits ground in lowest position,RGB value greater than 255,Chain stays intersect,Tube wall thickness exceeds radius,Down tube intersects front wheel
0,-66.108078,-64.120621,-516.410034,-548.616577,72.690781,-674.399719,-1651.818481,-673.098511,-523.669739,-11.148895,-999999808.0,136200.84375,-789.918762,-454.422852,36.228806,192.264648,221.241943
